In [ ]:
import pandas as pd
import numpy as np
import math

In [ ]:
def bin_target(df: pd.DataFrame, target_col: str) -> dict:
    """
    Discretise a target column using Scott's rule and return a mapping
    of bin_id -> list of row indices.

    Scott's rule:
        bin_width  h = 3.5 * std(x) * n^(-1/3)
        n_bins       = ceil((max(x) - min(x)) / h)

    Parameters
    ----------
    df         : input DataFrame
    target_col : name of the column to discretise

    Returns
    -------
    dict  {bin_id (int, 0-based): [row_index, ...]}
    """
    x = df[target_col].dropna().values
    n = len(x)

    if n < 2:
        raise ValueError("Need at least 2 non-null values to compute bins.")

    std = np.std(x, ddof=1)          # sample standard deviation
    if std == 0:
        raise ValueError("Target column has zero variance; cannot bin.")

    h       = 3.5 * std * (n ** (-1 / 3))          # Scott's bin width
    n_bins  = max(1, math.ceil((x.max() - x.min()) / h))

    # Cut into n_bins equal-width intervals; labels are 0-based integer bin IDs
    bin_series = pd.cut(
        df[target_col],
        bins=n_bins,
        labels=range(n_bins),
        include_lowest=True
    )

    bin_dict = {}
    for bin_id, group in df.groupby(bin_series, observed=True):
        bin_dict[int(bin_id)] = list(group.index)

    return bin_dict, n_bins

In [ ]:
# Sample DataFrame
np.random.seed(42)
df = pd.DataFrame({
    "feature_a": np.random.randn(100),
    "feature_b": np.random.rand(100),
    "target":    np.random.normal(loc=50, scale=15, size=100)
})

bin_map, n_bins = bin_target(df, target_col="target")

print(f"Scott's rule → {n_bins} bins\n")
for bin_id, indices in bin_map.items():
    print(f"  Bin {bin_id:>2d} ({len(indices):>3d} items): {indices[:5]}{'...' if len(indices) > 5 else ''}")